In [ ]:
# Notebook 5: PointNet++ XYZ-only APS-to-YCB-28 experiment

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import json
import time
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

PROJECT_DIR = Path("/content/drive/MyDrive/PointNet_APS_Project_V2")

SRC_DIR = PROJECT_DIR / "src"
METADATA_DIR = PROJECT_DIR / "metadata"

MODELS_XYZ_DIR = PROJECT_DIR / "models_final_xyz"
RESULTS_XYZ_DIR = PROJECT_DIR / "results_final_xyz"
FIGURES_XYZ_DIR = PROJECT_DIR / "figures_final_xyz"

RUN_LOGS_DIR = RESULTS_XYZ_DIR / "run_logs"
YCB_RESULTS_DIR = RESULTS_XYZ_DIR / "ycb28_results"
YCB_CM_DIR = RESULTS_XYZ_DIR / "ycb28_confusion_matrices"
STATS_DIR = RESULTS_XYZ_DIR / "statistics"
CURVES_DIR = FIGURES_XYZ_DIR / "training_curves"

for d in [
    MODELS_XYZ_DIR,
    RESULTS_XYZ_DIR,
    FIGURES_XYZ_DIR,
    RUN_LOGS_DIR,
    YCB_RESULTS_DIR,
    YCB_CM_DIR,
    STATS_DIR,
    CURVES_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.append(str(SRC_DIR))

from pointnetpp_aps_utils import (
    APSPointCloudDataset,
    PointNetPPClassifier,
    set_global_seed
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Project folder:", PROJECT_DIR)
print("XYZ model folder:", MODELS_XYZ_DIR)
print("XYZ results folder:", RESULTS_XYZ_DIR)

Mounted at /content/drive


In [ ]:
# Load APS manifest and YCB-28 bbox-normalised manifest

aps_manifest_path = METADATA_DIR / "dataset_manifest_v2.csv"
ycb_manifest_path = METADATA_DIR / "ycb28_pointcloud_manifest_bbox_norm.csv"

print("APS manifest exists:", aps_manifest_path.exists())
print("YCB manifest exists:", ycb_manifest_path.exists())

manifest_df = pd.read_csv(aps_manifest_path)
ycb_df = pd.read_csv(ycb_manifest_path)

print("\nAPS manifest shape:", manifest_df.shape)
print("YCB manifest shape:", ycb_df.shape)

print("\nAPS roles:")
display(manifest_df["role"].value_counts())

print("\nYCB class counts:")
display(ycb_df["class_name"].value_counts())

print("\nYCB object counts:")
display(ycb_df.groupby(["class_name", "object_name"]).size().reset_index(name="count"))

print("\nExpected YCB total: 560")
print("Actual YCB total:", len(ycb_df))

APS manifest exists: True
YCB manifest exists: True

APS manifest shape: (4173, 8)
YCB manifest shape: (560, 10)

APS roles:


,count
role,
test_error,600
test_clean,600
validation,600
test_error_double,600
optional_reference_only,591
train_clean_condition,591
train_error_condition,591



YCB class counts:


,count
class_name,
sphere,220
box,180
cylinder,160



YCB object counts:


,class_name,object_name,count
0,box,003_cracker_box,20
1,box,004_sugar_box,20
2,box,008_pudding_box,20
3,box,009_gelatin_box,20
4,box,010_potted_meat_can,20
5,box,026_sponge,20
6,box,036_wood_block,20
7,box,061_foam_brick,20
8,box,077_rubiks_cube,20
9,cylinder,001_chips_can,20



Expected YCB total: 560
Actual YCB total: 560


In [ ]:
# Prepare APS training/validation dataframes

train_clean_df = manifest_df[manifest_df["role"] == "train_clean_condition"].copy().reset_index(drop=True)
train_error_df = manifest_df[manifest_df["role"] == "train_error_condition"].copy().reset_index(drop=True)
val_df = manifest_df[manifest_df["role"] == "validation"].copy().reset_index(drop=True)

train_mixed_df = pd.concat([train_clean_df, train_error_df], ignore_index=True)

print("Training/validation sizes:")
print("Clean train:", len(train_clean_df))
print("Error train:", len(train_error_df))
print("Mixed train:", len(train_mixed_df))
print("Validation:", len(val_df))
print("YCB test:", len(ycb_df))

print("\nClean train class counts:")
display(train_clean_df["class_name"].value_counts())

print("\nError train class counts:")
display(train_error_df["class_name"].value_counts())

print("\nValidation class counts:")
display(val_df["class_name"].value_counts())

print("\nYCB test class counts:")
display(ycb_df["class_name"].value_counts())

Training/validation sizes:
Clean train: 591
Error train: 591
Mixed train: 1182
Validation: 600
YCB test: 560

Clean train class counts:


,count
class_name,
box,220
cylinder,190
sphere,181



Error train class counts:


,count
class_name,
box,220
cylinder,190
sphere,181



Validation class counts:


,count
class_name,
box,200
cylinder,200
sphere,200



YCB test class counts:


,count
class_name,
sphere,220
box,180
cylinder,160


In [4]:
# Copy required CSV files from Google Drive to local Colab storage
# This makes training/testing more stable and faster.

LOCAL_CACHE_DIR = Path("/content/pointnetpp_xyz_cache")
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def copy_dataframe_files_to_local(df, subset_name):
    df = df.copy().reset_index(drop=True)

    local_paths = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Copying {subset_name}"):
        src = Path(row["path"])

        if not src.exists():
            raise FileNotFoundError(f"Missing source file: {src}")

        class_name = row["class_name"]
        file_name = src.name

        dst = LOCAL_CACHE_DIR / subset_name / class_name / file_name
        dst.parent.mkdir(parents=True, exist_ok=True)

        if not dst.exists():
            shutil.copy2(src, dst)

        local_paths.append(str(dst))

    df["original_drive_path"] = df["path"]
    df["path"] = local_paths

    return df

train_clean_local_df = copy_dataframe_files_to_local(train_clean_df, "aps_clean_train")
train_error_local_df = copy_dataframe_files_to_local(train_error_df, "aps_error_train")
train_mixed_local_df = pd.concat([train_clean_local_df, train_error_local_df], ignore_index=True)
val_local_df = copy_dataframe_files_to_local(val_df, "aps_validation")
ycb_local_df = copy_dataframe_files_to_local(ycb_df, "ycb28_test")

print("Local copy complete.")
print("Clean train:", len(train_clean_local_df))
print("Error train:", len(train_error_local_df))
print("Mixed train:", len(train_mixed_local_df))
print("Validation:", len(val_local_df))
print("YCB test:", len(ycb_local_df))

print("\nSample local YCB path exists:")
print(Path(ycb_local_df.iloc[0]["path"]).exists())
print(ycb_local_df.iloc[0]["path"])

Copying ycb28_test: 100%|██████████| 560/560 [05:02<00:00,  1.85it/s]

Local copy complete.
Clean train: 591
Error train: 591
Mixed train: 1182
Validation: 600
YCB test: 560

Sample local YCB path exists:
True
/content/pointnetpp_xyz_cache/ycb28_test/box/003_cracker_box_sample_00.csv


In [5]:
# Experiment settings

CLASS_NAMES = ["box", "cylinder", "sphere"]

N_POINTS = 1000
USE_NORMALS = False          # Important: XYZ-only experiment
BATCH_SIZE = 16              # Safe for Colab T4/L4
NUM_WORKERS = 0

EPOCHS = 50
PATIENCE = 12                # Early stopping patience
LEARNING_RATE = 8e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05

# Pilot first: 3 runs per condition
RUN_IDS = [1, 2, 3, 4, 5, 6, 7, 8, 9]

TRAIN_CONDITIONS = {
    "clean_trained_xyz": train_clean_local_df,
    "error_trained_xyz": train_error_local_df,
    "mixed_trained_xyz": train_mixed_local_df,
}

BASE_SEEDS = {
    "clean_trained_xyz": 1100,
    "error_trained_xyz": 2100,
    "mixed_trained_xyz": 3100,
}

print("Using normals:", USE_NORMALS)
print("Epochs:", EPOCHS)
print("Runs:", RUN_IDS)
print("Training conditions:", list(TRAIN_CONDITIONS.keys()))

Using normals: False
Epochs: 50
Runs: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Training conditions: ['clean_trained_xyz', 'error_trained_xyz', 'mixed_trained_xyz']


In [6]:
# Dataset wrapper with augmentation for XYZ-only point clouds

class AugmentedPointCloudDataset(Dataset):
    def __init__(
        self,
        base_dataset,
        rotate=True,
        scale=True,
        jitter=True,
        shift=True,
        point_dropout=True,
        jitter_std=0.01,
        jitter_clip=0.03,
        dropout_ratio=0.05
    ):
        self.base_dataset = base_dataset
        self.rotate = rotate
        self.scale = scale
        self.jitter = jitter
        self.shift = shift
        self.point_dropout = point_dropout
        self.jitter_std = jitter_std
        self.jitter_clip = jitter_clip
        self.dropout_ratio = dropout_ratio

    def __len__(self):
        return len(self.base_dataset)

    def random_rotation_matrix(self):
        angles = torch.rand(3) * 2 * torch.pi
        rx, ry, rz = angles

        Rx = torch.tensor([
            [1, 0, 0],
            [0, torch.cos(rx), -torch.sin(rx)],
            [0, torch.sin(rx), torch.cos(rx)]
        ], dtype=torch.float32)

        Ry = torch.tensor([
            [torch.cos(ry), 0, torch.sin(ry)],
            [0, 1, 0],
            [-torch.sin(ry), 0, torch.cos(ry)]
        ], dtype=torch.float32)

        Rz = torch.tensor([
            [torch.cos(rz), -torch.sin(rz), 0],
            [torch.sin(rz), torch.cos(rz), 0],
            [0, 0, 1]
        ], dtype=torch.float32)

        return Rz @ Ry @ Rx

    def __getitem__(self, idx):
        points, label = self.base_dataset[idx]

        if not torch.is_tensor(points):
            points = torch.tensor(points, dtype=torch.float32)
        else:
            points = points.float()

        # APSPointCloudDataset returns [C, N], so for XYZ-only it is [3, 1000]
        # Convert to [N, 3] only for augmentation
        if points.shape[0] == 3:
            xyz = points.T.clone()       # [1000, 3]
        elif points.shape[-1] == 3:
            xyz = points.clone()         # already [1000, 3]
        else:
            raise ValueError(f"Unexpected point shape: {points.shape}")

        if self.rotate:
            R = self.random_rotation_matrix()
            xyz = xyz @ R.T

        if self.scale:
            scale_factor = torch.empty(1).uniform_(0.8, 1.2)
            xyz = xyz * scale_factor

        if self.shift:
            shift_vector = torch.empty(3).uniform_(-0.02, 0.02)
            xyz = xyz + shift_vector

        if self.jitter:
            noise = torch.clamp(
                self.jitter_std * torch.randn_like(xyz),
                -self.jitter_clip,
                self.jitter_clip
            )
            xyz = xyz + noise

        if self.point_dropout:
            dropout_mask = torch.rand(xyz.shape[0]) < self.dropout_ratio
            if dropout_mask.any():
                replacement_point = xyz[0].clone()
                xyz[dropout_mask] = replacement_point

        # Convert back to [3, 1000] because this PointNet++ model expects [B, C, N]
        xyz = xyz.T.contiguous()

        return xyz, label

In [7]:
# DataLoader helper functions

def make_balanced_sampler(df):
    labels = df["label"].astype(int).values
    class_counts = np.bincount(labels, minlength=3)

    class_weights = 1.0 / np.maximum(class_counts, 1)
    sample_weights = class_weights[labels]

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True
    )

    return sampler


def make_train_loader(train_df):
    base_dataset = APSPointCloudDataset(
        train_df,
        n_points=N_POINTS,
        use_normals=USE_NORMALS
    )

    aug_dataset = AugmentedPointCloudDataset(base_dataset)

    sampler = make_balanced_sampler(train_df)

    loader = DataLoader(
        aug_dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        drop_last=False
    )

    return loader


def make_eval_loader(eval_df):
    dataset = APSPointCloudDataset(
        eval_df,
        n_points=N_POINTS,
        use_normals=USE_NORMALS
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        drop_last=False
    )

    return loader


# Create validation and YCB-28 evaluation loaders

val_loader = make_eval_loader(val_local_df)
ycb_loader = make_eval_loader(ycb_local_df)

print("Validation samples:", len(val_local_df))
print("Validation batches:", len(val_loader))

print("YCB-28 samples:", len(ycb_local_df))
print("YCB-28 batches:", len(ycb_loader))

Validation samples: 600
Validation batches: 38
YCB-28 samples: 560
YCB-28 batches: 35


In [8]:
# Model, loss, evaluation helper functions

def create_xyz_model():
    model = PointNetPPClassifier(
        num_classes=3,
        normal_channel=False
    ).to(device)

    return model


def compute_class_weights(train_df):
    labels = train_df["label"].astype(int).values
    class_counts = np.bincount(labels, minlength=3)

    weights = 1.0 / np.maximum(class_counts, 1)
    weights = weights / weights.mean()

    return torch.tensor(weights, dtype=torch.float32).to(device)


def prepare_points_for_model(points):
    points = points.to(device).float()

    # Correct shape for this PointNet++ implementation: [B, C, N]
    # XYZ-only should be [B, 3, 1000]
    if points.ndim == 3 and points.shape[1] == 3:
        return points

    # If the batch is [B, 1000, 3], convert to [B, 3, 1000]
    if points.ndim == 3 and points.shape[-1] == 3:
        points = points.transpose(2, 1).contiguous()

    return points


def evaluate_model(model, loader, criterion=None):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    all_true = []
    all_pred = []
    all_conf = []

    with torch.no_grad():
        for points, labels in loader:
            labels = labels.to(device).long()
            points = prepare_points_for_model(points)

            logits = model(points)

            if isinstance(logits, tuple):
                logits = logits[0]

            if criterion is not None:
                loss = criterion(logits, labels)
                total_loss += loss.item() * labels.size(0)

            probs = F.softmax(logits, dim=1)
            conf, preds = probs.max(dim=1)

            total_samples += labels.size(0)
            all_true.extend(labels.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())
            all_conf.extend(conf.cpu().numpy())

    acc = accuracy_score(all_true, all_pred)
    avg_loss = total_loss / total_samples if criterion is not None else np.nan

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "true": np.array(all_true),
        "pred": np.array(all_pred),
        "confidence": np.array(all_conf)
    }

In [ ]:
# Quick shape test before training

test_loader = make_train_loader(train_clean_local_df)
points, labels = next(iter(test_loader))

print("Batch shape from loader:", points.shape)

points_prepared = prepare_points_for_model(points)
print("Batch shape after prepare_points_for_model:", points_prepared.shape)

model = create_xyz_model()
model.eval()

with torch.no_grad():
    logits = model(points_prepared)

if isinstance(logits, tuple):
    logits = logits[0]

print("Model output shape:", logits.shape)

Batch shape from loader: torch.Size([16, 3, 1000])
Batch shape after prepare_points_for_model: torch.Size([16, 3, 1000])
Model output shape: torch.Size([16, 3])


In [9]:
# Training function with validation checkpointing and early stopping

def train_one_xyz_model(train_condition, train_df, run_id):
    seed = BASE_SEEDS[train_condition] + run_id
    set_global_seed(seed)

    condition_model_dir = MODELS_XYZ_DIR / train_condition
    condition_model_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_path = condition_model_dir / f"{train_condition}_run_{run_id}_best.pth"
    history_path = RUN_LOGS_DIR / f"{train_condition}_run_{run_id}_history.csv"

    if checkpoint_path.exists() and history_path.exists():
        print(f"Checkpoint already exists, skipping training: {checkpoint_path}")
        history_df = pd.read_csv(history_path)
        return checkpoint_path, history_df

    train_loader = make_train_loader(train_df)

    model = create_xyz_model()

    class_weights = compute_class_weights(train_df)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=LABEL_SMOOTHING
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5
    )

    best_val_acc = -1.0
    best_val_loss = np.inf
    best_epoch = -1
    patience_counter = 0

    history = []

    start_time = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()

        train_loss_sum = 0.0
        train_samples = 0
        train_true = []
        train_pred = []

        for points, labels in train_loader:
            labels = labels.to(device).long()
            points = prepare_points_for_model(points)

            optimizer.zero_grad()

            logits = model(points)
            if isinstance(logits, tuple):
                logits = logits[0]

            loss = criterion(logits, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            probs = F.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)

            train_loss_sum += loss.item() * labels.size(0)
            train_samples += labels.size(0)

            train_true.extend(labels.detach().cpu().numpy())
            train_pred.extend(preds.detach().cpu().numpy())

        train_loss = train_loss_sum / train_samples
        train_acc = accuracy_score(train_true, train_pred)

        val_result = evaluate_model(model, val_loader, criterion=criterion)
        val_loss = val_result["loss"]
        val_acc = val_result["accuracy"]

        scheduler.step(val_loss)

        current_lr = optimizer.param_groups[0]["lr"]

        improved = False

        if val_acc > best_val_acc:
            improved = True
        elif val_acc == best_val_acc and val_loss < best_val_loss:
            improved = True

        if improved:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0

            torch.save({
                "model_state_dict": model.state_dict(),
                "train_condition": train_condition,
                "run_id": run_id,
                "seed": seed,
                "epoch": epoch,
                "best_val_accuracy": best_val_acc,
                "best_val_loss": best_val_loss,
                "use_normals": USE_NORMALS,
                "n_points": N_POINTS,
                "class_names": CLASS_NAMES,
                "settings": {
                    "epochs": EPOCHS,
                    "batch_size": BATCH_SIZE,
                    "learning_rate": LEARNING_RATE,
                    "weight_decay": WEIGHT_DECAY,
                    "label_smoothing": LABEL_SMOOTHING,
                    "patience": PATIENCE
                }
            }, checkpoint_path)

        else:
            patience_counter += 1

        history.append({
            "train_condition": train_condition,
            "run_id": run_id,
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "best_val_accuracy_so_far": best_val_acc,
            "best_val_loss_so_far": best_val_loss,
            "best_epoch_so_far": best_epoch,
            "learning_rate": current_lr,
            "patience_counter": patience_counter
        })

        print(
            f"{train_condition} run {run_id} | "
            f"Epoch {epoch:03d}/{EPOCHS} | "
            f"Train Acc {train_acc*100:.2f}% | "
            f"Val Acc {val_acc*100:.2f}% | "
            f"Train Loss {train_loss:.4f} | "
            f"Val Loss {val_loss:.4f} | "
            f"Best Val {best_val_acc*100:.2f}%"
        )

        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}")
            break

    elapsed_min = (time.time() - start_time) / 60.0

    history_df = pd.DataFrame(history)
    history_df["training_time_min"] = elapsed_min
    history_df.to_csv(history_path, index=False)

    print(f"Saved checkpoint: {checkpoint_path}")
    print(f"Saved history: {history_path}")
    print(f"Training time: {elapsed_min:.2f} min")

    torch.cuda.empty_cache()

    return checkpoint_path, history_df

In [ ]:
# Run XYZ-only pilot training

trained_records = []

for train_condition, train_df in TRAIN_CONDITIONS.items():
    for run_id in RUN_IDS:
        print("\n" + "="*80)
        print(f"Training {train_condition}, run {run_id}")
        print("="*80)

        checkpoint_path, history_df = train_one_xyz_model(
            train_condition=train_condition,
            train_df=train_df,
            run_id=run_id
        )

        trained_records.append({
            "train_condition": train_condition,
            "run_id": run_id,
            "checkpoint_path": str(checkpoint_path),
            "history_rows": len(history_df),
            "best_val_accuracy_percent": history_df["best_val_accuracy_so_far"].max() * 100,
            "best_epoch": int(history_df.loc[history_df["best_val_accuracy_so_far"].idxmax(), "epoch"])
        })

trained_df = pd.DataFrame(trained_records)
trained_path = STATS_DIR / "xyz_pilot_training_records.csv"
trained_df.to_csv(trained_path, index=False)

display(trained_df)
print("Saved:", trained_path)


Training clean_trained_xyz, run 1
Checkpoint already exists, skipping training: /content/drive/MyDrive/PointNet_APS_Project_V2/models_final_xyz/clean_trained_xyz/clean_trained_xyz_run_1_best.pth

Training clean_trained_xyz, run 2
Checkpoint already exists, skipping training: /content/drive/MyDrive/PointNet_APS_Project_V2/models_final_xyz/clean_trained_xyz/clean_trained_xyz_run_2_best.pth

Training clean_trained_xyz, run 3
Checkpoint already exists, skipping training: /content/drive/MyDrive/PointNet_APS_Project_V2/models_final_xyz/clean_trained_xyz/clean_trained_xyz_run_3_best.pth

Training clean_trained_xyz, run 4
clean_trained_xyz run 4 | Epoch 001/50 | Train Acc 67.68% | Val Acc 33.33% | Train Loss 0.7254 | Val Loss 1.6323 | Best Val 33.33%
clean_trained_xyz run 4 | Epoch 002/50 | Train Acc 75.97% | Val Acc 71.17% | Train Loss 0.6159 | Val Loss 0.7064 | Best Val 71.17%
clean_trained_xyz run 4 | Epoch 003/50 | Train Acc 83.25% | Val Acc 95.50% | Train Loss 0.4898 | Val Loss 0.3352 | 

,train_condition,run_id,checkpoint_path,history_rows,best_val_accuracy_percent,best_epoch
0,clean_trained_xyz,1,/content/drive/MyDrive/PointNet_APS_Project_V2...,50,100.0,23
1,clean_trained_xyz,2,/content/drive/MyDrive/PointNet_APS_Project_V2...,50,100.0,28
2,clean_trained_xyz,3,/content/drive/MyDrive/PointNet_APS_Project_V2...,48,100.0,20
3,clean_trained_xyz,4,/content/drive/MyDrive/PointNet_APS_Project_V2...,50,100.0,18
4,clean_trained_xyz,5,/content/drive/MyDrive/PointNet_APS_Project_V2...,48,100.0,21
5,clean_trained_xyz,6,/content/drive/MyDrive/PointNet_APS_Project_V2...,50,100.0,18
6,clean_trained_xyz,7,/content/drive/MyDrive/PointNet_APS_Project_V2...,50,100.0,22
7,clean_trained_xyz,8,/content/drive/MyDrive/PointNet_APS_Project_V2...,50,100.0,16
8,clean_trained_xyz,9,/content/drive/MyDrive/PointNet_APS_Project_V2...,50,100.0,27
9,error_trained_xyz,1,/content/drive/MyDrive/PointNet_APS_Project_V2...,50,100.0,24


Saved: /content/drive/MyDrive/PointNet_APS_Project_V2/results_final_xyz/statistics/xyz_pilot_training_records.csv


In [ ]:
# Check all trained XYZ-only checkpoints

checkpoint_records = []

for train_condition in TRAIN_CONDITIONS.keys():
    for run_id in RUN_IDS:
        ckpt_path = MODELS_XYZ_DIR / train_condition / f"{train_condition}_run_{run_id}_best.pth"

        checkpoint_records.append({
            "train_condition": train_condition,
            "run_id": run_id,
            "checkpoint_path": str(ckpt_path),
            "exists": ckpt_path.exists()
        })

checkpoint_df = pd.DataFrame(checkpoint_records)

display(checkpoint_df)

print("Checkpoint availability:")
display(checkpoint_df.groupby("train_condition")["exists"].sum())

print("\nMissing checkpoints:")
display(checkpoint_df[checkpoint_df["exists"] == False])

checkpoint_check_path = STATS_DIR / "xyz_checkpoint_availability.csv"
checkpoint_df.to_csv(checkpoint_check_path, index=False)

print("Saved checkpoint check:", checkpoint_check_path)

,train_condition,run_id,checkpoint_path,exists
0,clean_trained_xyz,1,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
1,clean_trained_xyz,2,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
2,clean_trained_xyz,3,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
3,clean_trained_xyz,4,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
4,clean_trained_xyz,5,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
5,clean_trained_xyz,6,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
6,clean_trained_xyz,7,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
7,clean_trained_xyz,8,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
8,clean_trained_xyz,9,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
9,error_trained_xyz,1,/content/drive/MyDrive/PointNet_APS_Project_V2...,True


Checkpoint availability:


,exists
train_condition,
clean_trained_xyz,9
error_trained_xyz,9
mixed_trained_xyz,9



Missing checkpoints:


,train_condition,run_id,checkpoint_path,exists


Saved checkpoint check: /content/drive/MyDrive/PointNet_APS_Project_V2/results_final_xyz/statistics/xyz_checkpoint_availability.csv


In [ ]:
# Prepare APS test loaders for XYZ-only models

print("Available roles in manifest:")
display(manifest_df["role"].value_counts())

# Automatically find APS test sets from the manifest
test_roles = sorted([r for r in manifest_df["role"].unique() if "test" in str(r).lower()])

print("\nDetected test roles:")
print(test_roles)

aps_test_datasets = {}

for role in test_roles:
    role_lower = role.lower()

    if "clean" in role_lower and "error" not in role_lower:
        test_name = "aps_clean_test"
    elif "double" in role_lower:
        test_name = "aps_error_double_test"
    elif "error" in role_lower:
        test_name = "aps_error_test"
    else:
        test_name = role

    df = manifest_df[manifest_df["role"] == role].copy().reset_index(drop=True)
    aps_test_datasets[test_name] = df

print("\nAPS test datasets detected:")
for name, df in aps_test_datasets.items():
    print(name, ":", len(df))
    display(df["class_name"].value_counts())

# Copy APS test files locally for stable evaluation
aps_test_local_datasets = {}

for test_name, df in aps_test_datasets.items():
    local_df = copy_dataframe_files_to_local(df, test_name)
    aps_test_local_datasets[test_name] = local_df

# Create loaders
aps_test_loaders = {}

for test_name, df in aps_test_local_datasets.items():
    aps_test_loaders[test_name] = make_eval_loader(df)

print("\nAPS test loaders ready:")
for name, loader in aps_test_loaders.items():
    print(name, "batches:", len(loader), "samples:", len(loader.dataset))

Available roles in manifest:


,count
role,
test_error,600
test_clean,600
validation,600
test_error_double,600
optional_reference_only,591
train_clean_condition,591
train_error_condition,591



Detected test roles:
['test_clean', 'test_error', 'test_error_double']

APS test datasets detected:
aps_clean_test : 600


,count
class_name,
box,200
cylinder,200
sphere,200


aps_error_test : 600


,count
class_name,
box,200
cylinder,200
sphere,200


aps_error_double_test : 600


,count
class_name,
box,200
cylinder,200
sphere,200


Copying aps_error_double_test: 100%|██████████| 600/600 [06:57<00:00,  1.44it/s]


APS test loaders ready:
aps_clean_test batches: 38 samples: 600
aps_error_test batches: 38 samples: 600
aps_error_double_test batches: 38 samples: 600


In [10]:
# Load XYZ-only checkpoint helper

def load_xyz_checkpoint(checkpoint_path):
    model = create_xyz_model()

    checkpoint = torch.load(checkpoint_path, map_location=device)

    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)

    model.eval()
    return model

In [ ]:
# Evaluate XYZ-only models on APS clean/error/error-double test sets

APS_XYZ_RESULTS_DIR = RESULTS_XYZ_DIR / "aps_test_results"
APS_XYZ_CM_DIR = RESULTS_XYZ_DIR / "aps_confusion_matrices"

APS_XYZ_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
APS_XYZ_CM_DIR.mkdir(parents=True, exist_ok=True)

aps_summary_records = []
aps_prediction_records = []

for _, row in checkpoint_df.iterrows():
    train_condition = row["train_condition"]
    run_id = int(row["run_id"])
    checkpoint_path = Path(row["checkpoint_path"])

    if not checkpoint_path.exists():
        print(f"Skipping missing checkpoint: {checkpoint_path}")
        continue

    print("\n" + "="*80)
    print(f"Loading model: {train_condition}, run {run_id}")
    print("="*80)

    model = load_xyz_checkpoint(checkpoint_path)

    for test_name, loader in aps_test_loaders.items():
        print(f"Evaluating on {test_name}...")

        result = evaluate_model(model, loader, criterion=None)

        y_true = result["true"]
        y_pred = result["pred"]
        y_conf = result["confidence"]

        acc = accuracy_score(y_true, y_pred)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

        cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
        cm_path = APS_XYZ_CM_DIR / f"{train_condition}_run_{run_id}_{test_name}_confusion_matrix.csv"
        cm_df.to_csv(cm_path)

        test_df = aps_test_local_datasets[test_name].copy().reset_index(drop=True)

        pred_df = test_df.copy()
        pred_df["true_label"] = y_true
        pred_df["pred_label"] = y_pred
        pred_df["true_class"] = [CLASS_NAMES[i] for i in y_true]
        pred_df["pred_class"] = [CLASS_NAMES[i] for i in y_pred]
        pred_df["confidence"] = y_conf
        pred_df["correct"] = pred_df["true_label"] == pred_df["pred_label"]
        pred_df["train_condition"] = train_condition
        pred_df["run_id"] = run_id
        pred_df["test_set"] = test_name

        pred_path = APS_XYZ_RESULTS_DIR / f"{train_condition}_run_{run_id}_{test_name}_predictions.csv"
        pred_df.to_csv(pred_path, index=False)

        class_acc = {}

        for class_id, class_name in enumerate(CLASS_NAMES):
            mask = y_true == class_id
            class_acc[class_name] = accuracy_score(y_true[mask], y_pred[mask]) if mask.sum() > 0 else np.nan

        aps_summary_records.append({
            "train_condition": train_condition,
            "run_id": run_id,
            "test_set": test_name,
            "test_accuracy": acc,
            "test_accuracy_percent": acc * 100,
            "correct_predictions": int((y_true == y_pred).sum()),
            "wrong_predictions": int((y_true != y_pred).sum()),
            "total_samples": len(y_true),
            "box_accuracy_percent": class_acc["box"] * 100,
            "cylinder_accuracy_percent": class_acc["cylinder"] * 100,
            "sphere_accuracy_percent": class_acc["sphere"] * 100,
            "confusion_matrix_path": str(cm_path),
            "prediction_path": str(pred_path)
        })

        aps_prediction_records.append(pred_df)

aps_xyz_summary_df = pd.DataFrame(aps_summary_records)

aps_xyz_summary_path = APS_XYZ_RESULTS_DIR / "pointnetpp_xyz_aps_test_summary_runs.csv"
aps_xyz_summary_df.to_csv(aps_xyz_summary_path, index=False)

all_aps_xyz_predictions_df = pd.concat(aps_prediction_records, ignore_index=True)
aps_xyz_predictions_path = APS_XYZ_RESULTS_DIR / "pointnetpp_xyz_aps_test_all_predictions_runs.csv"
all_aps_xyz_predictions_df.to_csv(aps_xyz_predictions_path, index=False)

print("\nSaved APS XYZ-only summary:")
print(aps_xyz_summary_path)

print("\nSaved APS XYZ-only all predictions:")
print(aps_xyz_predictions_path)

display(aps_xyz_summary_df)


Loading model: clean_trained_xyz, run 1
Evaluating on aps_clean_test...
Evaluating on aps_error_test...
Evaluating on aps_error_double_test...

Loading model: clean_trained_xyz, run 2
Evaluating on aps_clean_test...
Evaluating on aps_error_test...
Evaluating on aps_error_double_test...

Loading model: clean_trained_xyz, run 3
Evaluating on aps_clean_test...
Evaluating on aps_error_test...
Evaluating on aps_error_double_test...

Loading model: clean_trained_xyz, run 4
Evaluating on aps_clean_test...
Evaluating on aps_error_test...
Evaluating on aps_error_double_test...

Loading model: clean_trained_xyz, run 5
Evaluating on aps_clean_test...
Evaluating on aps_error_test...
Evaluating on aps_error_double_test...

Loading model: clean_trained_xyz, run 6
Evaluating on aps_clean_test...
Evaluating on aps_error_test...
Evaluating on aps_error_double_test...

Loading model: clean_trained_xyz, run 7
Evaluating on aps_clean_test...
Evaluating on aps_error_test...
Evaluating on aps_error_double_

,train_condition,run_id,test_set,test_accuracy,test_accuracy_percent,correct_predictions,wrong_predictions,total_samples,box_accuracy_percent,cylinder_accuracy_percent,sphere_accuracy_percent,confusion_matrix_path,prediction_path
0,clean_trained_xyz,1,aps_clean_test,1.000000,100.000000,600,0,600,100.0,100.0,100.0,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
1,clean_trained_xyz,1,aps_error_test,0.991667,99.166667,595,5,600,100.0,97.5,100.0,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
2,clean_trained_xyz,1,aps_error_double_test,0.568333,56.833333,341,259,600,99.0,42.0,29.5,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
3,clean_trained_xyz,2,aps_clean_test,1.000000,100.000000,600,0,600,100.0,100.0,100.0,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
4,clean_trained_xyz,2,aps_error_test,1.000000,100.000000,600,0,600,100.0,100.0,100.0,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,mixed_trained_xyz,8,aps_error_test,1.000000,100.000000,600,0,600,100.0,100.0,100.0,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
77,mixed_trained_xyz,8,aps_error_double_test,0.991667,99.166667,595,5,600,97.5,100.0,100.0,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
78,mixed_trained_xyz,9,aps_clean_test,1.000000,100.000000,600,0,600,100.0,100.0,100.0,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
79,mixed_trained_xyz,9,aps_error_test,1.000000,100.000000,600,0,600,100.0,100.0,100.0,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...


In [ ]:
# Evaluate XYZ-only PointNet++ models on YCB-28

ycb_summary_records = []
all_prediction_records = []

for _, row in checkpoint_df.iterrows():
    train_condition = row["train_condition"]
    run_id = int(row["run_id"])
    checkpoint_path = Path(row["checkpoint_path"])

    if not checkpoint_path.exists():
        print(f"Skipping missing checkpoint: {checkpoint_path}")
        continue

    print(f"Evaluating {train_condition}, run {run_id} on YCB-28...")

    model = load_xyz_checkpoint(checkpoint_path)
    result = evaluate_model(model, ycb_loader, criterion=None)

    y_true = result["true"]
    y_pred = result["pred"]
    y_conf = result["confidence"]

    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

    cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
    cm_path = YCB_CM_DIR / f"{train_condition}_run_{run_id}_ycb28_confusion_matrix.csv"
    cm_df.to_csv(cm_path)

    pred_df = ycb_local_df.copy().reset_index(drop=True)
    pred_df["true_label"] = y_true
    pred_df["pred_label"] = y_pred
    pred_df["true_class"] = [CLASS_NAMES[i] for i in y_true]
    pred_df["pred_class"] = [CLASS_NAMES[i] for i in y_pred]
    pred_df["confidence"] = y_conf
    pred_df["correct"] = pred_df["true_label"] == pred_df["pred_label"]
    pred_df["train_condition"] = train_condition
    pred_df["run_id"] = run_id
    pred_df["test_set"] = "ycb28"

    pred_path = YCB_RESULTS_DIR / f"{train_condition}_run_{run_id}_ycb28_predictions.csv"
    pred_df.to_csv(pred_path, index=False)

    class_acc = {}

    for class_id, class_name in enumerate(CLASS_NAMES):
        mask = y_true == class_id
        class_acc[class_name] = accuracy_score(y_true[mask], y_pred[mask]) if mask.sum() > 0 else np.nan

    ycb_summary_records.append({
        "train_condition": train_condition,
        "run_id": run_id,
        "test_set": "ycb28",
        "test_accuracy": acc,
        "test_accuracy_percent": acc * 100,
        "correct_predictions": int((y_true == y_pred).sum()),
        "wrong_predictions": int((y_true != y_pred).sum()),
        "total_samples": len(y_true),
        "box_accuracy_percent": class_acc["box"] * 100,
        "cylinder_accuracy_percent": class_acc["cylinder"] * 100,
        "sphere_accuracy_percent": class_acc["sphere"] * 100,
        "confusion_matrix_path": str(cm_path),
        "prediction_path": str(pred_path)
    })

    all_prediction_records.append(pred_df)

ycb_summary_df = pd.DataFrame(ycb_summary_records)

ycb_summary_path = YCB_RESULTS_DIR / "pointnetpp_xyz_aps_to_ycb28_summary_runs.csv"
ycb_summary_df.to_csv(ycb_summary_path, index=False)

all_ycb_predictions_df = pd.concat(all_prediction_records, ignore_index=True)
all_predictions_path = YCB_RESULTS_DIR / "pointnetpp_xyz_aps_to_ycb28_all_predictions_runs.csv"
all_ycb_predictions_df.to_csv(all_predictions_path, index=False)

print("Saved YCB summary:")
print(ycb_summary_path)

print("\nSaved all YCB predictions:")
print(all_predictions_path)

display(ycb_summary_df)

Evaluating clean_trained_xyz, run 1 on YCB-28...
Evaluating clean_trained_xyz, run 2 on YCB-28...
Evaluating clean_trained_xyz, run 3 on YCB-28...
Evaluating clean_trained_xyz, run 4 on YCB-28...
Evaluating clean_trained_xyz, run 5 on YCB-28...
Evaluating clean_trained_xyz, run 6 on YCB-28...
Evaluating clean_trained_xyz, run 7 on YCB-28...
Evaluating clean_trained_xyz, run 8 on YCB-28...
Evaluating clean_trained_xyz, run 9 on YCB-28...
Evaluating error_trained_xyz, run 1 on YCB-28...
Evaluating error_trained_xyz, run 2 on YCB-28...
Evaluating error_trained_xyz, run 3 on YCB-28...
Evaluating error_trained_xyz, run 4 on YCB-28...
Evaluating error_trained_xyz, run 5 on YCB-28...
Evaluating error_trained_xyz, run 6 on YCB-28...
Evaluating error_trained_xyz, run 7 on YCB-28...
Evaluating error_trained_xyz, run 8 on YCB-28...
Evaluating error_trained_xyz, run 9 on YCB-28...
Evaluating mixed_trained_xyz, run 1 on YCB-28...
Evaluating mixed_trained_xyz, run 2 on YCB-28...
Evaluating mixed_tra

,train_condition,run_id,test_set,test_accuracy,test_accuracy_percent,correct_predictions,wrong_predictions,total_samples,box_accuracy_percent,cylinder_accuracy_percent,sphere_accuracy_percent,confusion_matrix_path,prediction_path
0,clean_trained_xyz,1,ycb28,0.794643,79.464286,445,115,560,88.888889,53.750,90.454545,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
1,clean_trained_xyz,2,ycb28,0.837500,83.750000,469,91,560,83.333333,74.375,90.909091,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
2,clean_trained_xyz,3,ycb28,0.819643,81.964286,459,101,560,85.000000,66.250,90.909091,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
3,clean_trained_xyz,4,ycb28,0.816071,81.607143,457,103,560,92.222222,56.875,90.909091,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
4,clean_trained_xyz,5,ycb28,0.841071,84.107143,471,89,560,88.888889,69.375,90.909091,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
5,clean_trained_xyz,6,ycb28,0.867857,86.785714,486,74,560,87.777778,80.000,90.909091,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
6,clean_trained_xyz,7,ycb28,0.835714,83.571429,468,92,560,82.222222,75.000,90.909091,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
7,clean_trained_xyz,8,ycb28,0.816071,81.607143,457,103,560,90.000000,59.375,90.909091,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
8,clean_trained_xyz,9,ycb28,0.835714,83.571429,468,92,560,93.888889,61.875,90.909091,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
9,error_trained_xyz,1,ycb28,0.875000,87.500000,490,70,560,91.111111,78.750,90.909091,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...


In [13]:
# Save final Notebook 5 status metadata

checkpoint_count = len(list(MODELS_XYZ_DIR.rglob("*.pth")))

notebook5_status = {
    "notebook": "05_xyz_only_adamw_training_ycb28",
    "purpose": "Train XYZ-only PointNet++ models on APS and evaluate them on APS and YCB-28",
    "input_type": "XYZ only",
    "use_normals": USE_NORMALS,
    "n_points": N_POINTS,
    "epochs": EPOCHS,
    "run_ids": RUN_IDS,
    "train_conditions": list(TRAIN_CONDITIONS.keys()),
    "checkpoint_count": checkpoint_count,
    "expected_checkpoint_count": 27,

    "aps_summary_path": str(
        RESULTS_XYZ_DIR
        / "aps_test_results"
        / "pointnetpp_xyz_aps_test_summary_runs.csv"
    ),

    "aps_predictions_path": str(
        RESULTS_XYZ_DIR
        / "aps_test_results"
        / "pointnetpp_xyz_aps_test_all_predictions_runs.csv"
    ),

    "ycb_summary_path": str(
        RESULTS_XYZ_DIR
        / "ycb28_results"
        / "pointnetpp_xyz_aps_to_ycb28_summary_runs.csv"
    ),

    "ycb_predictions_path": str(
        RESULTS_XYZ_DIR
        / "ycb28_results"
        / "pointnetpp_xyz_aps_to_ycb28_all_predictions_runs.csv"
    ),

    "run_logs_dir": str(RUN_LOGS_DIR),
    "model_dir": str(MODELS_XYZ_DIR),
    "results_dir": str(RESULTS_XYZ_DIR)
}

status_path = STATS_DIR / "05_xyz_only_adamw_training_ycb28_status.json"

with open(status_path, "w") as f:
    json.dump(notebook5_status, f, indent=4)

print("Notebook 5 completed.")
print(f"Checkpoints found: {checkpoint_count}/27")
print("Saved status file:")
print(status_path)

notebook5_status

Notebook 5 completed.
Checkpoints found: 27/27
Saved status file:
/content/drive/MyDrive/PointNet_APS_Project_V2/results_final_xyz/statistics/05_xyz_only_adamw_training_ycb28_status.json


{'notebook': '05_xyz_only_adamw_training_ycb28',
 'purpose': 'Train XYZ-only PointNet++ models on APS and evaluate them on APS and YCB-28',
 'input_type': 'XYZ only',
 'use_normals': False,
 'n_points': 1000,
 'epochs': 50,
 'run_ids': [1, 2, 3, 4, 5, 6, 7, 8, 9],
 'train_conditions': ['clean_trained_xyz',
  'error_trained_xyz',
  'mixed_trained_xyz'],
 'checkpoint_count': 27,
 'expected_checkpoint_count': 27,
 'aps_summary_path': '/content/drive/MyDrive/PointNet_APS_Project_V2/results_final_xyz/aps_test_results/pointnetpp_xyz_aps_test_summary_runs.csv',
 'aps_predictions_path': '/content/drive/MyDrive/PointNet_APS_Project_V2/results_final_xyz/aps_test_results/pointnetpp_xyz_aps_test_all_predictions_runs.csv',
 'ycb_summary_path': '/content/drive/MyDrive/PointNet_APS_Project_V2/results_final_xyz/ycb28_results/pointnetpp_xyz_aps_to_ycb28_summary_runs.csv',
 'ycb_predictions_path': '/content/drive/MyDrive/PointNet_APS_Project_V2/results_final_xyz/ycb28_results/pointnetpp_xyz_aps_to_ycb28